In [1]:
# ============================================================
# CELL 1 — Import
# ============================================================

import numpy as np
import pandas as pd
import torch

from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pykeen.triples import TriplesFactory
from pykeen.models import TransE
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.sampling import BasicNegativeSampler

from torch.optim import Adam


# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cuda


In [2]:
# ============================================================
# CELL 2 — Load Dataset
# ============================================================

TRAIN_PATH = "../data/processed/train.csv"
TEST_PATH = "../data/processed/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

assert train_df.shape == (64000, 8)
assert test_df.shape == (16000, 8)

TRAIN: (64000, 8)
TEST : (16000, 8)


In [3]:
# ============================================================
# CELL 3 — Definisi Feature
# ============================================================

TARGET = "dev_status"

FEATURE_COLUMNS = [
    "weight",
    "height",
    "age",
    "gross_motor",
    "fine_motor_adaptive",
    "language",
    "personal_social"
]

CATEGORICAL_FEATURES = [
    "gross_motor",
    "fine_motor_adaptive",
    "language",
    "personal_social"
]

NUMERIC_FEATURES = [
    "weight",
    "height",
    "age"
]

assert TARGET not in FEATURE_COLUMNS
assert len(FEATURE_COLUMNS) == 7

print("Features:", FEATURE_COLUMNS)
print("Target:", TARGET)

Features: ['weight', 'height', 'age', 'gross_motor', 'fine_motor_adaptive', 'language', 'personal_social']
Target: dev_status


In [4]:
# ============================================================
# CELL 4 — Parameter CatBoost
# ============================================================

BASELINE_PARAMS = {
    "learning_rate": 0.1,
    "depth": 6,
    "iterations": 200,
    "random_seed": 42
}

print("Parameter CatBoost:")
print(BASELINE_PARAMS)

Parameter CatBoost:
{'learning_rate': 0.1, 'depth': 6, 'iterations': 200, 'random_seed': 42}


In [5]:
# ============================================================
# CELL 5 — Konfigurasi KGE
# ============================================================

RELATION_MAP = {
    "weight": "has_weight",
    "height": "has_height",
    "age": "has_age",
    "gross_motor": "has_gross_motor",
    "fine_motor_adaptive": "has_fine_motor_adaptive",
    "language": "has_language",
    "personal_social": "has_personal_social"
}

KGE_DIM = 100
KGE_MARGIN = 1.0
KGE_EPOCHS = 100
KGE_BATCH_SIZE = 256

In [6]:
# ============================================================
# CELL 6 — Fungsi Membuat KG
# ============================================================

def build_kg(train_df):

    triples = []

    train_df = train_df.reset_index(drop=True)

    for i, (_, row) in enumerate(
        train_df.iterrows(),
        start=1
    ):

        child = f"child_{i:06d}"

        for feature in FEATURE_COLUMNS:

            relation = RELATION_MAP[feature]

            if feature in NUMERIC_FEATURES:

                tail = (
                    f"{feature}_"
                    f"{float(row[feature]):g}"
                )

            else:

                tail = (
                    f"{feature}_"
                    f"{row[feature]}"
                )

            triples.append(
                (
                    child,
                    relation,
                    tail
                )
            )

    triples = np.asarray(
        triples,
        dtype=str
    )

    triples_factory = (
        TriplesFactory.from_labeled_triples(
            triples
        )
    )

    return triples_factory

In [7]:
# ============================================================
# CELL 7 — Fungsi Training TransE
# ============================================================

def train_transe(train_df):

    # Reproducibility
    np.random.seed(42)
    torch.manual_seed(42)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    triples_factory = build_kg(
        train_df
    )

    model = TransE(
        triples_factory=triples_factory,
        embedding_dim=KGE_DIM,
        loss=MarginRankingLoss(
            margin=KGE_MARGIN
        )
    ).to(DEVICE)

    optimizer = Adam(
        params=model.get_grad_params()
    )

    training_loop = SLCWATrainingLoop(
        model=model,
        triples_factory=triples_factory,
        optimizer=optimizer,
        negative_sampler=BasicNegativeSampler
    )

    training_loop.train(
        triples_factory=triples_factory,
        num_epochs=KGE_EPOCHS,
        batch_size=KGE_BATCH_SIZE
    )

    entity_embeddings = (
        model.entity_representations[0]()
        .detach()
        .cpu()
        .numpy()
    )

    relation_embeddings = (
        model.relation_representations[0]()
        .detach()
        .cpu()
        .numpy()
    )

    return (
        triples_factory,
        entity_embeddings,
        relation_embeddings
    )

In [8]:
# ============================================================
# CELL 8 — Fungsi Mengubah Data menjadi KGE
# ============================================================

def create_kge_encoder(
    triples_factory,
    entity_embeddings,
    relation_embeddings,
    train_df
):

    entity_to_id = triples_factory.entity_to_id
    relation_to_id = triples_factory.relation_to_id

    numeric_entities = {}

    for feature in NUMERIC_FEATURES:

        prefix = f"{feature}_"
        values = []

        for entity in entity_to_id:

            if entity.startswith(prefix):

                try:

                    value = float(
                        entity[len(prefix):]
                    )

                    values.append(
                        (value, entity)
                    )

                except ValueError:
                    pass

        values.sort(
            key=lambda x: x[0]
        )

        numeric_entities[feature] = values

    def get_tail_entity(
        feature,
        value
    ):

        # Categorical
        if feature not in NUMERIC_FEATURES:

            entity = (
                f"{feature}_{value}"
            )

            if entity not in entity_to_id:

                raise ValueError(
                    f"Entity tidak ditemukan: "
                    f"{entity}"
                )

            return entity

        # Numeric
        numeric_value = float(value)

        candidates = (
            numeric_entities[feature]
        )

        if not candidates:

            raise ValueError(
                f"Tidak ada entity numeric "
                f"untuk {feature}"
            )

        exact_entity = (
            f"{feature}_{numeric_value:g}"
        )

        if exact_entity in entity_to_id:

            return exact_entity

        # nearest value dari train
        return min(
            candidates,
            key=lambda x:
            abs(x[0] - numeric_value)
        )[1]

    def encode_row(row):

        vectors = []

        for feature in FEATURE_COLUMNS:

            relation = (
                RELATION_MAP[feature]
            )

            tail_entity = (
                get_tail_entity(
                    feature,
                    row[feature]
                )
            )

            tail_id = (
                entity_to_id[
                    tail_entity
                ]
            )

            relation_id = (
                relation_to_id[
                    relation
                ]
            )

            child_vector = (
                entity_embeddings[tail_id]
                -
                relation_embeddings[
                    relation_id
                ]
            )

            vectors.append(
                child_vector
            )

        return np.mean(
            vectors,
            axis=0
        )

    def encode_dataframe(df):

        return np.vstack([
            encode_row(row)
            for _, row in df.iterrows()
        ]).astype(
            np.float32
        )

    return encode_dataframe

In [9]:
# ============================================================
# CELL 9 — Pisahkan TRAIN dan TEST
# ============================================================

X_train = (
    train_df[
        FEATURE_COLUMNS
    ]
    .reset_index(drop=True)
)

y_train = (
    train_df[
        TARGET
    ]
    .reset_index(drop=True)
)

X_test = (
    test_df[
        FEATURE_COLUMNS
    ]
    .reset_index(drop=True)
)

y_test = (
    test_df[
        TARGET
    ]
    .reset_index(drop=True)
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (64000, 7)
y_train: (64000,)
X_test : (16000, 7)
y_test : (16000,)


In [10]:
# ============================================================
# CELL 10 — Training KGE menggunakan TRAIN
# ============================================================

print("Training KGE menggunakan TRAIN...")

tf, entity_emb, relation_emb = (
    train_transe(
        X_train
    )
)

print("KGE training selesai.")

Training KGE menggunakan TRAIN...


No random seed is specified. This may lead to non-reproducible results.


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/1.75k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/1.75k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/1.75k [00:00<?, ?batch/s]

... (94 outputs left out)

Training batches on cuda:0:   0%|          | 0.00/1.75k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/1.75k [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/1.75k [00:00<?, ?batch/s]

KGE training selesai.


In [11]:
# ============================================================
# CELL 11 — Membuat Encoder KGE
# ============================================================

encoder = create_kge_encoder(
    tf,
    entity_emb,
    relation_emb,
    X_train
)

print("KGE encoder berhasil dibuat.")

KGE encoder berhasil dibuat.


In [12]:
# ============================================================
# CELL 12 — Membuat KGE TRAIN dan TEST
# ============================================================

kge_train = encoder(
    X_train
)

kge_test = encoder(
    X_test
)

print("KGE Train:", kge_train.shape)
print("KGE Test :", kge_test.shape)

KGE Train: (64000, 100)
KGE Test : (16000, 100)


In [13]:
# ============================================================
# CELL 13 — Feature Integration
# ============================================================

KGE_COLUMNS = [
    f"kge_{i}"
    for i in range(1, 101)
]


def integrate_features(
    original_df,
    kge_array
):

    original_features = (
        original_df[
            FEATURE_COLUMNS
        ]
        .reset_index(drop=True)
    )

    kge_features = pd.DataFrame(
        kge_array,
        columns=KGE_COLUMNS
    )

    result = pd.concat(
        [
            original_features,
            kge_features
        ],
        axis=1
    )

    assert result.shape[1] == 107
    assert TARGET not in result.columns
    assert result.isna().sum().sum() == 0

    return result

In [14]:
# ============================================================
# CELL 14 — Integrasi Feature TRAIN dan TEST
# ============================================================

X_train_final = integrate_features(
    X_train,
    kge_train
)

X_test_final = integrate_features(
    X_test,
    kge_test
)

print(
    "X_train_final:",
    X_train_final.shape
)

print(
    "X_test_final :",
    X_test_final.shape
)

X_train_final: (64000, 107)
X_test_final : (16000, 107)


In [15]:
# ============================================================
# CELL 15 — Check Feature
# ============================================================

assert X_train_final.shape == (
    64000,
    107
)

assert X_test_final.shape == (
    16000,
    107
)

assert TARGET not in X_train_final.columns
assert TARGET not in X_test_final.columns

assert len(y_train) == 64000
assert len(y_test) == 16000

assert (
    X_train_final.isna().sum().sum()
    == 0
)

assert (
    X_test_final.isna().sum().sum()
    == 0
)

print("✅ Feature check berhasil.")

✅ Feature check berhasil.


In [16]:
# ============================================================
# CELL 16 — Train CatBoost
# ============================================================

final_model = CatBoostClassifier(

    learning_rate=
        BASELINE_PARAMS[
            "learning_rate"
        ],

    depth=
        BASELINE_PARAMS[
            "depth"
        ],

    iterations=
        BASELINE_PARAMS[
            "iterations"
        ],

    random_seed=42,

    boosting_type="Ordered",

    grow_policy="SymmetricTree",

    loss_function="Logloss",

    verbose=False,

    allow_writing_files=False,

    thread_count=-1
)

final_model.fit(

    X_train_final,

    y_train,

    cat_features=
        CATEGORICAL_FEATURES
)

print(
    "✅ CatBoost selesai dilatih."
)

print(
    "Jumlah data training:",
    len(X_train_final)
)

✅ CatBoost selesai dilatih.
Jumlah data training: 64000


In [17]:
# ============================================================
# CELL 17 — Prediksi TEST
# ============================================================

prediction = (
    final_model
    .predict(X_test_final)
    .ravel()
)

print(
    "Jumlah data prediction:",
    len(prediction)
)

Jumlah data prediction: 16000


In [18]:
# ============================================================
# CELL 18 — Evaluasi
# ============================================================

accuracy = accuracy_score(
    y_test,
    prediction
)

precision = precision_score(
    y_test,
    prediction,
    pos_label="suspect",
    zero_division=0
)

recall = recall_score(
    y_test,
    prediction,
    pos_label="suspect",
    zero_division=0
)

f1 = f1_score(
    y_test,
    prediction,
    pos_label="suspect",
    zero_division=0
)

print("===== HASIL EVALUASI TEST =====")

print(
    "Accuracy :",
    round(accuracy, 4)
)

print(
    "Precision:",
    round(precision, 4)
)

print(
    "Recall   :",
    round(recall, 4)
)

print(
    "F1       :",
    round(f1, 4)
)

===== HASIL EVALUASI TEST =====
Accuracy : 0.9738
Precision: 0.9756
Recall   : 0.992
F1       : 0.9837


In [19]:
# ============================================================
# CELL 19 — Classification Report
# ============================================================

print(
    classification_report(
        y_test,
        prediction,
        labels=[
            "normal",
            "suspect"
        ],
        target_names=[
            "normal",
            "suspect"
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

      normal       0.97      0.90      0.93      3204
     suspect       0.98      0.99      0.98     12796

    accuracy                           0.97     16000
   macro avg       0.97      0.95      0.96     16000
weighted avg       0.97      0.97      0.97     16000



In [20]:
# ============================================================
# CELL 20 — Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    prediction,
    labels=[
        "normal",
        "suspect"
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual normal",
        "Actual suspect"
    ],
    columns=[
        "Predicted normal",
        "Predicted suspect"
    ]
)

display(cm_df)

,Predicted normal,Predicted suspect
Actual normal,2886,318
Actual suspect,102,12694


In [21]:
# ============================================================
# CELL 21 — Feature Importance
# ============================================================

feature_importance = (
    final_model
    .get_feature_importance()
)

feature_importance_df = pd.DataFrame({

    "feature":
        X_train_final.columns,

    "importance":
        feature_importance

})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    feature_importance_df
)

,feature,importance
0,gross_motor,20.876525
1,personal_social,19.985715
2,language,19.168922
3,fine_motor_adaptive,18.364028
4,kge_32,1.385136
...,...,...
102,kge_88,0.023962
103,height,0.023669
104,weight,0.022732
105,age,0.009495


In [22]:
# ============================================================
# CELL 22 — Final Check
# ============================================================

print("======================================")
print("FINAL CHECK")
print("======================================")

print(
    "TRAIN:",
    X_train_final.shape
)

print(
    "TEST :",
    X_test_final.shape
)

print(
    "KGE Dimension:",
    KGE_DIM
)

print(
    "CatBoost Parameters:"
)

for key, value in BASELINE_PARAMS.items():

    print(
        f"{key}: {value}"
    )

print("\nEvaluation TEST:")

print(
    "Accuracy :",
    round(
        accuracy,
        4
    )
)

print(
    "Precision:",
    round(
        precision,
        4
    )
)

print(
    "Recall   :",
    round(
        recall,
        4
    )
)

print(
    "F1       :",
    round(
        f1,
        4
    )
)

print("\nLeakage Check:")

print(
    "✅ Tidak menggunakan K-Fold"
)

print(
    "✅ Tidak menggunakan StratifiedKFold"
)

print(
    "✅ KGE dilatih menggunakan TRAIN"
)

print(
    "✅ TEST hanya digunakan untuk evaluasi"
)

print(
    "✅ dev_status tidak menjadi feature"
)

print(
    "✅ Tidak ada penyimpanan model"
)

print(
    "✅ Tidak ada penyimpanan hasil"
)

print(
    "✅ Tidak membaca feature hasil penyimpanan"
)

print(
    "\n✅ FINAL CHECK SELESAI"
)

FINAL CHECK
TRAIN: (64000, 107)
TEST : (16000, 107)
KGE Dimension: 100
CatBoost Parameters:
learning_rate: 0.1
depth: 6
iterations: 200
random_seed: 42

Evaluation TEST:
Accuracy : 0.9738
Precision: 0.9756
Recall   : 0.992
F1       : 0.9837

Leakage Check:
✅ Tidak menggunakan K-Fold
✅ Tidak menggunakan StratifiedKFold
✅ KGE dilatih menggunakan TRAIN
✅ TEST hanya digunakan untuk evaluasi
✅ dev_status tidak menjadi feature
✅ Tidak ada penyimpanan model
✅ Tidak ada penyimpanan hasil
✅ Tidak membaca feature hasil penyimpanan

✅ FINAL CHECK SELESAI
